# Stretch aware melspace gated fusion training

This notebook is part of the thesis notebook set.
It uses the shared prepared 70/30 speech/music split directory: `/content/drive/MyDrive/master/prepared_datasets/audio_70speech_30music_v1/splits`.

Purpose:
- This notebook tests stretch-aware gated fusion in mel space.
- Checkpoint prefixes and manual checkpoint paths are configuration fields, not fixed thesis assumptions.
- The model-specific training or evaluation logic is kept from the original notebook unless the configuration depended on an old data split.


In [ ]:

# =========================
# Installs / imports
# =========================
import os, sys, json, time, math, random, shutil, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio

from torch.utils.data import Dataset, DataLoader
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)

In [ ]:

# =========================
# Repro / config
# =========================
SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DRIVE_ROOT = Path("/content/drive/MyDrive/master")
PREPARED_DATASET_DIR = DRIVE_ROOT / "prepared_datasets" / "audio_70speech_30music_v1"
SPLIT_DIR = PREPARED_DATASET_DIR / "splits"
assert DRIVE_ROOT.exists(), f"Missing DRIVE_ROOT: {DRIVE_ROOT}"

MODE = "A1HYBRID"
RUN_NAME = "stretch_aware_melspace_gated_fusion"
RUN_TAG = f"{RUN_NAME}_{time.strftime('%Y%m%d_%H%M%S')}"
CKPT_DIR = DRIVE_ROOT / "checkpoints_runs" / RUN_TAG
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------
# Main idea
# -------------------------------------------------------
# Stretch-aware mel-space gated fusion:
#
#   1) Compute normal-hop mel anchors with HOP=256.
#   2) Compute a higher-time-resolution mel target with target_hop = HOP * speed.
#      Example: speed=0.5 -> target_hop=128 -> about twice as many mel frames.
#   3) Interpolate the normal-hop anchors onto the target grid.
#   4) Train a gated fusion model to refine inserted frames on that target grid.
#
# Final listening output is high-resolution mel -> BigVGAN -> stretched waveform.
# The complex-STFT teacher is used only as a guide; the final renderer remains BigVGAN.

# audio / dataset
SR = 22050
SEG_S = 2.0
BATCH = 1              # high-res target grids are longer; keep safe for long run
P_SPEECH = 0.70
NUM_WORKERS = 2
PIN_MEMORY = (device == "cuda")

# training
STEPS = 30000
SAVE_EVERY = 2500
LOG_EVERY = 100
RUN_SMOKE_TEST_BEFORE_TRAINING = True

# mel frontend (BigVGAN-compatible defaults)
N_MELS = 80
N_FFT = 1024
HOP = 256
WIN = 1024
FMIN = 0
FMAX = 8000
CENTER_MEL = False

# complex teacher STFT setup
CPLX_N_FFT = 1024
CPLX_HOP = 256
CPLX_WIN = 1024
CPLX_CENTER = True

# Stretch-aware training speeds.
# speed=0.5 means 0.5x playback speed / about 2x duration.
# speed=0.75 means about 1.33x duration.
# I recommend starting with 0.5 and 0.75 only; 1.0 is not needed for the main artifact.
STRETCH_SPEED_CHOICES = (0.5, 0.75)
MIN_TARGET_HOP = 64

# fusion model
G_BASE = 24
G_GROUPS = 8
USE_MASK = True
GATE_TEMP = 2.2
USE_ALPHA_CHANNEL = True  # important: tells model where inserted frame lies between anchors

# optimizer
LR_G = 1.2e-4
WEIGHT_DECAY = 1e-4
RESUME_LR_SCALE = 0.75

# losses
# This run targets inserted-frame slow-motion behavior.
# HF + temporal-difference matter more here because repeated inserted-frame errors become audible.
LAMBDA_RECON = 22.0
LAMBDA_HF = 85.0
LAMBDA_TDIFF = 32.0
LAMBDA_ANCHOR_RECON = 2.0   # weak preservation pressure on original anchor frames
LAMBDA_DELTA = 0.012
LAMBDA_GATE = 0.0008        # slightly lower so gate can open more than v50
LAMBDA_GATE_ALIGN = 4.5
LAMBDA_ROUNDTRIP_HF = 0.0   # keep OFF by default for stability; can turn on after a successful run

HF_START_FRAC = 0.40
HF_END_GAIN = 8.0
HF_RAMP_POWER = 2.0

MASK_DILATE = 3
TDIFF_MASK_DILATE = 3
PRIOR_RADIUS = 3
PRIOR_BOUNDARY_GAIN = 2.2
PRIOR_HF_POWER = 1.20

# optional BigVGAN roundtrip branch
USE_ROUNDTRIP_HF = False
ROUNDTRIP_EVERY = 2
ROUNDTRIP_SUBBATCH = 1

# -------------------------------------------------------
# Teacher checkpoints
# -------------------------------------------------------
# Recommended:
#   - mel teacher: clearest overall 2D mel model
#   - complex teacher: best artifact-reducing complex STFT U-Net
#
# If auto-detection chooses the wrong run, paste exact .pt paths manually.
MEL_TEACHER_CKPT = None
MEL_TEACHER_PREFIX = None  # Example: "teacher_run_prefix"

COMPLEX_TEACHER_CKPT = None
COMPLEX_TEACHER_PREFIX = None  # Example: "teacher_run_prefix"

# optional resume from this exact family only
RESUME_PATH = None
AUTO_RESUME_SAME_FAMILY = False

BIGVGAN_MODEL_ID = "nvidia/bigvgan_v2_22khz_80band_fmax8k_256x"
BIGVGAN_USE_CUDA_KERNEL = False

# Quick post-training listening example
EXACT_LISTEN_SPLIT_NAME  = "speech_test.txt"
EXACT_LISTEN_SPLIT_INDEX = 4
EVAL_SEG_S               = 6.0
EVAL_STRETCH_SPEEDS      = (0.5, 0.75)

RUN_CONFIG = dict(
    seed=SEED,
    mode=MODE,
    run_name=RUN_NAME,
    sr=SR,
    segment_s=SEG_S,
    batch=BATCH,
    p_speech=P_SPEECH,
    mel=dict(n_fft=N_FFT, hop=HOP, win=WIN, n_mels=N_MELS, fmin=FMIN, fmax=FMAX, center=CENTER_MEL),
    complex_stft=dict(n_fft=CPLX_N_FFT, hop=CPLX_HOP, win=CPLX_WIN, center=CPLX_CENTER),
    stretch_speeds=list(STRETCH_SPEED_CHOICES),
    use_alpha_channel=USE_ALPHA_CHANNEL,
    g_arch="HybridFusionUNet2D_stretchaware_melspace_gate_alpha",
    g_base=G_BASE,
    g_groups=G_GROUPS,
    gate_temp=GATE_TEMP,
    losses=dict(
        lambda_recon=LAMBDA_RECON,
        lambda_hf=LAMBDA_HF,
        lambda_tdiff=LAMBDA_TDIFF,
        lambda_anchor_recon=LAMBDA_ANCHOR_RECON,
        lambda_delta=LAMBDA_DELTA,
        lambda_gate=LAMBDA_GATE,
        lambda_gate_align=LAMBDA_GATE_ALIGN,
        lambda_roundtrip_hf=LAMBDA_ROUNDTRIP_HF,
        hf_start_frac=HF_START_FRAC,
        hf_end_gain=HF_END_GAIN,
        hf_ramp_power=HF_RAMP_POWER,
        mask_dilate=MASK_DILATE,
        tdiff_mask_dilate=TDIFF_MASK_DILATE,
        prior_radius=PRIOR_RADIUS,
        prior_boundary_gain=PRIOR_BOUNDARY_GAIN,
        prior_hf_power=PRIOR_HF_POWER,
        use_roundtrip_hf=USE_ROUNDTRIP_HF,
        roundtrip_every=ROUNDTRIP_EVERY,
        roundtrip_subbatch=ROUNDTRIP_SUBBATCH,
    ),
    teachers=dict(
        mel_teacher_ckpt=str(MEL_TEACHER_CKPT) if MEL_TEACHER_CKPT is not None else None,
        mel_teacher_prefix=MEL_TEACHER_PREFIX,
        complex_teacher_ckpt=str(COMPLEX_TEACHER_CKPT) if COMPLEX_TEACHER_CKPT is not None else None,
        complex_teacher_prefix=COMPLEX_TEACHER_PREFIX,
    ),
    resume=dict(
        resume_path=str(RESUME_PATH) if RESUME_PATH is not None else None,
        auto_resume_same_family=AUTO_RESUME_SAME_FAMILY,
        resume_lr_scale=RESUME_LR_SCALE,
    ),
)
(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
print("RUN_NAME:", RUN_NAME)
print("CKPT_DIR:", CKPT_DIR)
print("STRETCH_SPEED_CHOICES:", STRETCH_SPEED_CHOICES)


In [ ]:

# =========================
# BigVGAN repo / imports
# =========================
BIGVGAN_REPO = Path("/content/BigVGAN")
if not BIGVGAN_REPO.exists():
    subprocess.run(["git","clone","https://github.com/NVIDIA/BigVGAN.git","/content/BigVGAN"], check=True)

if str(BIGVGAN_REPO) not in sys.path:
    sys.path.insert(0, str(BIGVGAN_REPO))

import bigvgan
from meldataset import mel_spectrogram as bigvgan_mel_spectrogram

print("BigVGAN repo ready at:", BIGVGAN_REPO)

bigvgan_G = None

def ensure_bigvgan_loaded():
    global bigvgan_G
    if bigvgan_G is not None:
        return bigvgan_G

    print("Loading BigVGAN:", BIGVGAN_MODEL_ID)
    bigvgan_G = bigvgan.BigVGAN._from_pretrained(
        model_id=BIGVGAN_MODEL_ID,
        revision="main",
        cache_dir=None,
        force_download=False,
        proxies=None,
        resume_download=False,
        local_files_only=False,
        token=None,
        map_location=str(device),
        strict=False,
        use_cuda_kernel=BIGVGAN_USE_CUDA_KERNEL,
    ).to(device).eval()

    for p in bigvgan_G.parameters():
        p.requires_grad_(False)

    return bigvgan_G

def wav_to_bigvgan_mel_cfg(wav_bt: torch.Tensor, hop_size: int = HOP, center: bool = CENTER_MEL):
    # wav_bt: (B,T)
    if wav_bt.ndim == 1:
        wav_bt = wav_bt.unsqueeze(0)
    return bigvgan_mel_spectrogram(
        wav_bt,
        n_fft=N_FFT,
        num_mels=N_MELS,
        sampling_rate=SR,
        hop_size=int(hop_size),
        win_size=WIN,
        fmin=FMIN,
        fmax=FMAX,
        center=center,
    )

def wav_to_bigvgan_mel(wav_bt: torch.Tensor):
    return wav_to_bigvgan_mel_cfg(wav_bt, hop_size=HOP, center=CENTER_MEL)

def mel_to_wave_bigvgan(mel_bt: torch.Tensor):
    Gv = ensure_bigvgan_loaded()
    y = Gv(mel_bt)
    return y.squeeze(1) if y.ndim == 3 and y.shape[1] == 1 else y

# Optional preload sanity check
# Gv = ensure_bigvgan_loaded()
# with torch.no_grad():
#     y = Gv(torch.zeros(1, N_MELS, 10, device=device))
# print("BigVGAN dummy output shape:", tuple(y.shape))


In [ ]:

# =========================
# Local cache for wav files
# =========================
DRIVE_SPLIT_DIR = SPLIT_DIR
LOCAL_DATA_ROOT = Path("/content/audio_70speech_30music_v1_cache")
LOCAL_SPLIT_DIR = Path("/content/audio_70speech_30music_v1_splits_local")

LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)
assert SPLIT_DIR.exists(), f"Missing prepared split directory: {SPLIT_DIR}"

LISTS = ["speech_train.txt", "music_train.txt", "speech_val.txt", "music_val.txt", "speech_test.txt", "music_test.txt"]

def _read_lines(p: Path):
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [ln for ln in lines if ln]

def to_local_path(p: Path) -> Path:
    p = Path(p)
    for root in [DRIVE_ROOT, Path("/content/drive")]:
        try:
            rel = p.relative_to(root)
            return LOCAL_DATA_ROOT / rel
        except Exception:
            pass
    return LOCAL_DATA_ROOT / "flat" / p.name

all_files = []
for name in LISTS:
    src_list = DRIVE_SPLIT_DIR / name
    assert src_list.exists(), f"Missing split list: {src_list}"
    all_files += [Path(x) for x in _read_lines(src_list)]

all_files = sorted(set(all_files))
print("Unique audio files referenced by splits:", len(all_files))

copied = 0
skipped = 0
for src in all_files:
    dst = to_local_path(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        skipped += 1
        continue
    shutil.copy2(src, dst)
    copied += 1

print(f"Copied {copied} files (skipped {skipped} already present).")

for name in LISTS:
    src_list = DRIVE_SPLIT_DIR / name
    src_paths = [Path(x) for x in _read_lines(src_list)]
    dst_paths = [str(to_local_path(p)) for p in src_paths]
    (LOCAL_SPLIT_DIR / name).write_text("\n".join(dst_paths) + "\n", encoding="utf-8")

SPLIT_DIR = LOCAL_SPLIT_DIR
RUN_CONFIG["split_dir"] = str(SPLIT_DIR)
RUN_CONFIG["local_data_root"] = str(LOCAL_DATA_ROOT)
(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")

print("Using local cached wav paths.")


In [ ]:

# =========================
# Dataset / loaders
# =========================
def read_list(p: Path):
    assert p.exists(), f"Missing list: {p}"
    lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
    return [Path(ln) for ln in lines if ln]

def safe_load_wav_mono(path: Path, target_sr: int) -> torch.Tensor:
    path = Path(path)
    try:
        wav, sr = torchaudio.load(str(path))
        wav = wav.mean(dim=0)
    except Exception:
        data, sr = sf.read(str(path), dtype="float32", always_2d=True)
        wav = torch.from_numpy(data.T).mean(dim=0)

    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    wav = wav.to(torch.float32)
    wav = wav / (wav.abs().max() + 1e-8)
    wav = (0.95 * wav).clamp(-1.0, 1.0)
    return wav

class FileListDataset(Dataset):
    def __init__(self, list_path: Path, sr: int, segment_s: float):
        self.paths = read_list(list_path)
        self.sr = sr
        self.seg_len = int(sr * segment_s)
        assert len(self.paths) > 0, f"Empty list: {list_path}"

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        w = safe_load_wav_mono(self.paths[idx], self.sr)
        if w.numel() < self.seg_len:
            w = F.pad(w, (0, self.seg_len - w.numel()))
        max_start = w.numel() - self.seg_len
        start = random.randint(0, max_start) if max_start > 0 else 0
        return w[start:start+self.seg_len]

speech_train = SPLIT_DIR / "speech_train.txt"
music_train  = SPLIT_DIR / "music_train.txt"
speech_val   = SPLIT_DIR / "speech_val.txt"
music_val    = SPLIT_DIR / "music_val.txt"

ds_speech = FileListDataset(speech_train, SR, SEG_S)
ds_music  = FileListDataset(music_train,  SR, SEG_S)

dl_speech = DataLoader(
    ds_speech, batch_size=BATCH, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)
dl_music = DataLoader(
    ds_music, batch_size=BATCH, shuffle=True,
    num_workers=NUM_WORKERS, drop_last=True, pin_memory=PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

def infinite(dl):
    while True:
        for b in dl:
            yield b

it_speech = infinite(dl_speech)
it_music = infinite(dl_music)

def next_mixed_batch():
    b = next(it_speech) if random.random() < P_SPEECH else next(it_music)
    return b.to(device)

print("speech files:", len(ds_speech), "music files:", len(ds_music), "P_SPEECH:", P_SPEECH)


In [ ]:

# =========================
# Helpers: masks, weights, stretch-aware interpolation
# =========================
def build_freq_weights(n_bins: int, start_frac: float, end_gain: float, power: float, device=None):
    idx = torch.linspace(0.0, 1.0, steps=n_bins, device=device)
    ramp = ((idx - start_frac).clamp_min(0.0) / max(1e-6, 1.0 - start_frac)) ** power
    w = 1.0 + (end_gain - 1.0) * ramp
    return w.view(1, n_bins, 1)

HF_FREQ_WEIGHTS = build_freq_weights(N_MELS, HF_START_FRAC, HF_END_GAIN, HF_RAMP_POWER, device=device)

def dilate_mask_time(mask: torch.Tensor, radius: int):
    if radius <= 0:
        return mask
    return F.max_pool2d(mask, kernel_size=(1, 2 * radius + 1), stride=1, padding=(0, radius))

def expand_mask(mask: torch.Tensor, n_bins: int, radius: int = 0):
    # mask: (B,1,1,T) -> (B,n_bins,T)
    m = dilate_mask_time(mask, radius)
    return m[:, 0].expand(-1, n_bins, -1)

def masked_mean(x: torch.Tensor, mask_ft: torch.Tensor, freq_weights=None, eps: float = 1e-8):
    z = x.abs()
    if freq_weights is not None:
        z = z * freq_weights
        denom = (mask_ft * freq_weights).sum()
    else:
        denom = mask_ft.sum()
    return (z * mask_ft).sum() / (denom + eps)

def masked_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)
    return masked_mean(pred - tgt, m, freq_weights=freq_weights)

def anchor_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None):
    # Penalize anchor changes weakly. Anchors are frames with mask=0.
    m_insert = expand_mask(mask, pred.shape[1], radius=0)
    m_anchor = (1.0 - m_insert).clamp(0.0, 1.0)
    return masked_mean(pred - tgt, m_anchor, freq_weights=freq_weights)

def masked_tdiff_l1(pred: torch.Tensor, tgt: torch.Tensor, mask: torch.Tensor, freq_weights=None, mask_dilate: int = 0):
    dp = pred[..., 1:] - pred[..., :-1]
    dt = tgt[..., 1:] - tgt[..., :-1]
    m = expand_mask(mask, pred.shape[1], radius=mask_dilate)[..., 1:]
    return masked_mean(dp - dt, m, freq_weights=freq_weights)

def stft_complex_cfg(wav_bt: torch.Tensor, hop: int, center: bool = CPLX_CENTER):
    window = torch.hann_window(CPLX_WIN, device=wav_bt.device)
    return torch.stft(
        wav_bt,
        n_fft=CPLX_N_FFT,
        hop_length=int(hop),
        win_length=CPLX_WIN,
        window=window,
        center=center,
        return_complex=True,
    )

def istft_complex_cfg(X_bft: torch.Tensor, hop: int, length: int, center: bool = CPLX_CENTER):
    window = torch.hann_window(CPLX_WIN, device=X_bft.device)
    try:
        return torch.istft(
            X_bft,
            n_fft=CPLX_N_FFT,
            hop_length=int(hop),
            win_length=CPLX_WIN,
            window=window,
            center=center,
            length=length,
        )
    except RuntimeError as e:
        # Avoid the Hann-window COLA issue if center=False sneaks in.
        if "window overlap add" in str(e) and not center:
            return torch.istft(
                X_bft,
                n_fft=CPLX_N_FFT,
                hop_length=int(hop),
                win_length=CPLX_WIN,
                window=window,
                center=True,
                length=length,
            )
        raise

def stft_complex(wav_bt: torch.Tensor):
    return stft_complex_cfg(wav_bt, hop=CPLX_HOP, center=CPLX_CENTER)

def istft_complex(X_bft: torch.Tensor, length: int):
    return istft_complex_cfg(X_bft, hop=CPLX_HOP, length=length, center=CPLX_CENTER)

def _linear_resample_real(seq_bft: torch.Tensor, pos_t: torch.Tensor):
    """Linearly resample a real tensor (B,F,T) at floating frame positions pos_t."""
    B, Freq, T = seq_bft.shape
    pos = pos_t.to(seq_bft.device).clamp(0.0, max(0.0, T - 1.0))
    left = torch.floor(pos).long().clamp(0, T - 1)
    right = (left + 1).clamp(0, T - 1)
    alpha = (pos - left.float()).clamp(0.0, 1.0)
    idx_l = left.view(1, 1, -1).expand(B, Freq, -1)
    idx_r = right.view(1, 1, -1).expand(B, Freq, -1)
    vl = torch.gather(seq_bft, 2, idx_l)
    vr = torch.gather(seq_bft, 2, idx_r)
    out = (1.0 - alpha.view(1,1,-1)) * vl + alpha.view(1,1,-1) * vr
    return out, alpha

def _linear_resample_complex(seq_bft: torch.Tensor, pos_t: torch.Tensor):
    real, alpha = _linear_resample_real(seq_bft.real, pos_t)
    imag, _ = _linear_resample_real(seq_bft.imag, pos_t)
    return torch.complex(real, imag), alpha

def make_stretch_mask_and_alpha(alpha_t: torch.Tensor, batch_size: int, eps: float = 1e-5):
    # alpha=0 means this target frame lands exactly on a normal-hop anchor frame.
    # alpha>0 means inserted/fractional frame.
    mask_t = (alpha_t > eps).to(torch.float32)
    mask = mask_t.view(1,1,1,-1).expand(batch_size, 1, 1, -1).contiguous()
    alpha_ch = alpha_t.view(1,1,-1).expand(batch_size, 1, -1).contiguous()
    return mask, alpha_ch

def choose_stretch_speed():
    return float(random.choice(list(STRETCH_SPEED_CHOICES)))

def speed_to_hop(speed: float, base_hop: int):
    return max(MIN_TARGET_HOP, int(round(float(speed) * int(base_hop))))

def make_stretch_pair(wav_bt: torch.Tensor, speed: float = None):
    """Builds a supervised proxy for actual inserted-frame slow motion.

    Normal-hop anchors are interpolated onto a denser target grid. The target is the
    mel/STFT computed directly at the denser hop. For speed=0.5, target_hop=128 and
    the target grid contains halfway frames between the normal HOP=256 anchors.
    """
    if speed is None:
        speed = choose_stretch_speed()
    speed = float(speed)
    target_hop = speed_to_hop(speed, HOP)
    target_cplx_hop = speed_to_hop(speed, CPLX_HOP)

    B = wav_bt.shape[0]

    # Mel anchors and target on denser grid.
    mel_anchor = wav_to_bigvgan_mel_cfg(wav_bt, hop_size=HOP, center=CENTER_MEL)
    mel_real = wav_to_bigvgan_mel_cfg(wav_bt, hop_size=target_hop, center=CENTER_MEL)
    Tt = mel_real.shape[-1]
    pos_mel = torch.arange(Tt, device=wav_bt.device, dtype=torch.float32) * (target_hop / float(HOP))
    mel_interp, alpha_mel = _linear_resample_real(mel_anchor, pos_mel)
    mask_mel, alpha_ch = make_stretch_mask_and_alpha(alpha_mel, B)

    # Complex STFT anchors and target on denser grid.
    X_anchor = stft_complex_cfg(wav_bt, hop=CPLX_HOP, center=CPLX_CENTER)
    X_real = stft_complex_cfg(wav_bt, hop=target_cplx_hop, center=CPLX_CENTER)
    Tc = X_real.shape[-1]
    pos_cplx = torch.arange(Tc, device=wav_bt.device, dtype=torch.float32) * (target_cplx_hop / float(CPLX_HOP))
    X_interp, alpha_cplx = _linear_resample_complex(X_anchor, pos_cplx)
    mask_cplx, alpha_cplx_ch = make_stretch_mask_and_alpha(alpha_cplx, B)

    return {
        "wav": wav_bt,
        "speed": speed,
        "target_hop": int(target_hop),
        "target_cplx_hop": int(target_cplx_hop),
        "mel_anchor": mel_anchor,
        "mel_real": mel_real,
        "mel_interp": mel_interp,
        "mask_mel": mask_mel,
        "alpha_mel": alpha_ch,
        "X_real": X_real,
        "X_interp": X_interp,
        "mask_cplx": mask_cplx,
        "alpha_cplx": alpha_cplx_ch,
    }

def make_hf_prior(mask_mel: torch.Tensor, n_mels: int, radius: int = PRIOR_RADIUS):
    # Prior = inserted region + boundary ring, weighted toward upper mel bins.
    base = expand_mask(mask_mel, n_mels, radius=0)
    wide = expand_mask(mask_mel, n_mels, radius=radius)
    boundary = (wide - base).clamp_min(0.0)

    hf = HF_FREQ_WEIGHTS.expand(wide.shape[0], -1, wide.shape[-1])
    hf_norm = ((hf - 1.0) / max(1e-6, (HF_END_GAIN - 1.0))).clamp(0.0, 1.0)
    hf_shape = hf_norm ** PRIOR_HF_POWER

    region = (base + PRIOR_BOUNDARY_GAIN * boundary).clamp(0.0, 1.0 + PRIOR_BOUNDARY_GAIN)
    return (region * hf_shape).clamp(0.0, 1.0 + PRIOR_BOUNDARY_GAIN)

def make_gate_target(mel_teacher_hat: torch.Tensor, mel_complex_hat: torch.Tensor, mel_real: torch.Tensor, prior: torch.Tensor):
    # Encourage gate to open where the complex teacher is closer to the stretch target.
    err_mel = (mel_teacher_hat - mel_real).abs()
    err_cplx = (mel_complex_hat - mel_real).abs()
    gain = (err_mel - err_cplx).clamp_min(0.0)
    gain = gain * prior
    den = gain.amax(dim=(1,2), keepdim=True).clamp_min(1e-6)
    target = (gain / den).clamp(0.0, 1.0)
    return target ** 0.75


In [ ]:

# =========================
# Model defs: mel teacher, complex teacher, fusion model
# =========================
def _valid_groups(ch: int, requested: int):
    for g in [requested, 8, 4, 2, 1]:
        if ch % g == 0:
            return g
    return 1

class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p),
            nn.GroupNorm(g, out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        self.net = nn.Sequential(
            ConvGNAct(in_ch, out_ch, groups=groups),
            ConvGNAct(out_ch, out_ch, groups=groups),
        )
    def forward(self, x):
        return self.net(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=(2,2), groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            ConvGNAct(out_ch, out_ch, groups=groups),
        )
    def forward(self, x):
        return self.net(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, groups=8):
        super().__init__()
        self.fuse = DoubleConv(in_ch + skip_ch, out_ch, groups=groups)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip], dim=1)
        return self.fuse(x)

class MelRefinerUNet2D(nn.Module):
    def __init__(self, n_mels, base=24, use_mask=True, groups=8):
        super().__init__()
        self.use_mask = use_mask
        c_in = 1 + (1 if use_mask else 0)

        self.enc1 = DoubleConv(c_in,       base,     groups=groups)
        self.enc2 = DownBlock(base,        base*2,   stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2,      base*4,   stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4,      base*4,   stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4,     base*4,   groups=groups)

        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base,   groups=groups)

        self.out_conv = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)

    def forward(self, mel_interp: torch.Tensor, mask: torch.Tensor):
        x = mel_interp.unsqueeze(1)
        if self.use_mask:
            mask2d = mask.unsqueeze(2).expand(-1, 1, mel_interp.shape[1], -1)
            x = torch.cat([x, mask2d], dim=1)

        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        u0 = u0 + s1
        delta = self.out_conv(u0).squeeze(1)
        return delta

class ComplexConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, groups=8):
        super().__init__()
        g = _valid_groups(out_ch, groups)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class ComplexSTFTUNet2D(nn.Module):
    def __init__(self, in_ch=3, out_ch=2, base=24, groups=8):
        super().__init__()
        self.enc1 = ComplexConvGNAct(in_ch, base, groups)
        self.enc2 = ComplexConvGNAct(base, base*2, groups)
        self.enc3 = ComplexConvGNAct(base*2, base*4, groups)
        self.bot  = ComplexConvGNAct(base*4, base*8, groups)
        self.dec3 = ComplexConvGNAct(base*8 + base*4, base*4, groups)
        self.dec2 = ComplexConvGNAct(base*4 + base*2, base*2, groups)
        self.dec1 = ComplexConvGNAct(base*2 + base, base, groups)
        self.out = nn.Conv2d(base, out_ch, kernel_size=3, padding=1)
        self.pool = nn.AvgPool2d(kernel_size=2)

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.enc2(self.pool(x1))
        x3 = self.enc3(self.pool(x2))
        xb = self.bot(self.pool(x3))

        y3 = F.interpolate(xb, size=x3.shape[-2:], mode="bilinear", align_corners=False)
        y3 = self.dec3(torch.cat([y3, x3], dim=1))
        y2 = F.interpolate(y3, size=x2.shape[-2:], mode="bilinear", align_corners=False)
        y2 = self.dec2(torch.cat([y2, x2], dim=1))
        y1 = F.interpolate(y2, size=x1.shape[-2:], mode="bilinear", align_corners=False)
        y1 = self.dec1(torch.cat([y1, x1], dim=1))
        return self.out(y1)

def pack_complex_input(X_interp: torch.Tensor, mask: torch.Tensor):
    xr = X_interp.real.unsqueeze(1)
    xi = X_interp.imag.unsqueeze(1)
    xm = mask.expand(-1, 1, X_interp.shape[1], -1)
    return torch.cat([xr, xi, xm], dim=1)

class HybridFusionUNet2D(nn.Module):
    '''
    Input channels:
      mel_interp, mel_teacher, mel_complex, prior, mask, alpha

    alpha is the fractional position between anchors. For example, alpha=0.5 means
    the frame lies halfway between two original normal-hop anchor frames.
    '''
    def __init__(self, in_ch=6, base=24, groups=8):
        super().__init__()
        self.enc1 = DoubleConv(in_ch,       base,     groups=groups)
        self.enc2 = DownBlock(base,         base*2,   stride=(2,2), groups=groups)
        self.enc3 = DownBlock(base*2,       base*4,   stride=(2,2), groups=groups)
        self.enc4 = DownBlock(base*4,       base*4,   stride=(1,2), groups=groups)
        self.bot  = DoubleConv(base*4,      base*4,   groups=groups)

        self.up3 = UpBlock(base*4, base*4, base*4, groups=groups)
        self.up2 = UpBlock(base*4, base*4, base*2, groups=groups)
        self.up1 = UpBlock(base*2, base*2, base,   groups=groups)

        self.gate_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)
        self.delta_head = nn.Conv2d(base, 1, kernel_size=3, padding=1)

        nn.init.zeros_(self.gate_head.weight); nn.init.zeros_(self.gate_head.bias)
        nn.init.zeros_(self.delta_head.weight); nn.init.zeros_(self.delta_head.bias)

    def forward(self, mel_interp, mel_teacher, mel_complex, prior, mask, alpha):
        mask_ft = mask.expand(-1, mel_interp.shape[1], -1)
        alpha_ft = alpha.expand(-1, mel_interp.shape[1], -1)
        x = torch.stack([mel_interp, mel_teacher, mel_complex, prior, mask_ft, alpha_ft], dim=1)
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bot(s4)
        u3 = self.up3(b, s4)
        u2 = self.up2(u3, s3)
        u1 = self.up1(u2, s2)
        u0 = F.interpolate(u1, size=s1.shape[-2:], mode="bilinear", align_corners=False)
        u0 = u0 + s1
        gate_logits = self.gate_head(u0).squeeze(1)
        gate  = torch.sigmoid(GATE_TEMP * gate_logits)
        delta = self.delta_head(u0).squeeze(1)
        return gate, delta

G = HybridFusionUNet2D(in_ch=6, base=G_BASE, groups=G_GROUPS).to(device)
print("Stretch-aware fusion G params (M):", sum(p.numel() for p in G.parameters()) / 1e6)


In [ ]:

# =========================
# Load frozen teachers
# =========================
def find_latest_ckpt_by_prefix(prefix: str, root: Path = None):
    root = root or (DRIVE_ROOT / "checkpoints_runs")
    cands = sorted(root.glob(f"{prefix}_*/latest.pt"))
    if not cands:
        cands = sorted(root.glob(f"*{prefix}*/latest.pt"))
    if not cands:
        return None
    # timestamped folders sort lexicographically, but mtime fallback also works
    cands = sorted(cands, key=lambda p: (str(p.parent.name), p.stat().st_mtime))
    return cands[-1]

def get_run_config(ck):
    return ck.get("run_config", ck.get("config", {}))

def get_generator_state(ck):
    for key in ["G", "generator", "model", "state_dict"]:
        if key in ck:
            return ck[key]
    raise KeyError(f"Could not find generator state in checkpoint. Keys: {list(ck.keys())[:20]}")

# mel teacher auto-detect if needed
if MEL_TEACHER_CKPT is None:
    MEL_TEACHER_CKPT = None  # Example: Path("/content/drive/MyDrive/master/checkpoints_runs/<teacher_run_name>/latest.pt")
    assert MEL_TEACHER_CKPT is not None, f"Could not auto-detect mel teacher with prefix: {MEL_TEACHER_PREFIX}"

MEL_TEACHER_CKPT = None  # Example: Path("/content/drive/MyDrive/master/checkpoints_runs/<teacher_run_name>/latest.pt")
assert MEL_TEACHER_CKPT.exists(), f"Missing MEL_TEACHER_CKPT: {MEL_TEACHER_CKPT}"
ck_mel = torch.load(str(MEL_TEACHER_CKPT), map_location="cpu")
rc_mel = get_run_config(ck_mel)
mel_base = int(rc_mel.get("g_base", rc_mel.get("G_BASE", rc_mel.get("G_base", 24))))
mel_groups = int(rc_mel.get("g_groups", rc_mel.get("G_GROUPS", rc_mel.get("G_groups", 8))))
mel_use_mask = bool(rc_mel.get("G_use_mask", rc_mel.get("g_use_mask", True)))

mel_teacher = MelRefinerUNet2D(n_mels=N_MELS, base=mel_base, use_mask=mel_use_mask, groups=mel_groups).to(device)
state_mel = get_generator_state(ck_mel)
try:
    mel_teacher.load_state_dict(state_mel, strict=True)
except Exception as e:
    print("Strict mel teacher load failed; retrying strict=False.")
    print("Reason:", repr(e))
    missing, unexpected = mel_teacher.load_state_dict(state_mel, strict=False)
    print("Missing keys:", missing[:10], "..." if len(missing) > 10 else "")
    print("Unexpected keys:", unexpected[:10], "..." if len(unexpected) > 10 else "")
mel_teacher.eval()
for p in mel_teacher.parameters():
    p.requires_grad_(False)

# complex teacher auto-detect if needed
if COMPLEX_TEACHER_CKPT is None:
    COMPLEX_TEACHER_CKPT = None  # Example: Path("/content/drive/MyDrive/master/checkpoints_runs/<teacher_run_name>/latest.pt")
    assert COMPLEX_TEACHER_CKPT is not None, f"Could not auto-detect complex teacher with prefix: {COMPLEX_TEACHER_PREFIX}"

COMPLEX_TEACHER_CKPT = None  # Example: Path("/content/drive/MyDrive/master/checkpoints_runs/<teacher_run_name>/latest.pt")
assert COMPLEX_TEACHER_CKPT.exists(), f"Missing COMPLEX_TEACHER_CKPT: {COMPLEX_TEACHER_CKPT}"
ck_cplx = torch.load(str(COMPLEX_TEACHER_CKPT), map_location="cpu")
rc_cplx = get_run_config(ck_cplx)
cplx_base = int(rc_cplx.get("g_base", rc_cplx.get("G_BASE", rc_cplx.get("G_base", 24))))
cplx_groups = int(rc_cplx.get("g_groups", rc_cplx.get("G_GROUPS", rc_cplx.get("G_groups", 8))))
cplx_in = int(rc_cplx.get("g_in_ch", rc_cplx.get("G_IN_CH", 3)))
cplx_out = int(rc_cplx.get("g_out_ch", rc_cplx.get("G_OUT_CH", 2)))

complex_teacher = ComplexSTFTUNet2D(in_ch=cplx_in, out_ch=cplx_out, base=cplx_base, groups=cplx_groups).to(device)
state_cplx = get_generator_state(ck_cplx)
try:
    complex_teacher.load_state_dict(state_cplx, strict=True)
except Exception as e:
    print("Strict complex teacher load failed; retrying strict=False.")
    print("Reason:", repr(e))
    missing, unexpected = complex_teacher.load_state_dict(state_cplx, strict=False)
    print("Missing keys:", missing[:10], "..." if len(missing) > 10 else "")
    print("Unexpected keys:", unexpected[:10], "..." if len(unexpected) > 10 else "")
complex_teacher.eval()
for p in complex_teacher.parameters():
    p.requires_grad_(False)

RUN_CONFIG["teachers"]["mel_teacher_ckpt"] = str(MEL_TEACHER_CKPT)
RUN_CONFIG["teachers"]["complex_teacher_ckpt"] = str(COMPLEX_TEACHER_CKPT)
(CKPT_DIR / "run_config.json").write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")

print("Loaded mel teacher    :", MEL_TEACHER_CKPT)
print("Loaded complex teacher:", COMPLEX_TEACHER_CKPT)


In [ ]:

# =========================
# Teacher forward helpers
# =========================
@torch.no_grad()
def mel_teacher_forward(mel_interp: torch.Tensor, mask_mel: torch.Tensor):
    # Teacher was trained on normal-hop mel, but convolutional architecture accepts arbitrary T.
    delta = mel_teacher(mel_interp, mask_mel[:, :, 0, :])
    mask_ft = mask_mel[:, 0].expand(-1, mel_interp.shape[1], -1)
    return mel_interp + delta * mask_ft

@torch.no_grad()
def complex_teacher_forward_stretch(X_interp: torch.Tensor, mask_cplx: torch.Tensor, wav_len: int, target_cplx_hop: int, target_mel_hop: int):
    # Run the frozen complex-STFT teacher on the stretch target STFT grid.
    x_in = pack_complex_input(X_interp, mask_cplx)
    delta = complex_teacher(x_in)
    dX = torch.complex(delta[:, 0], delta[:, 1])
    mask_ft = mask_cplx[:, 0].expand(-1, X_interp.shape[1], -1)
    X_hat = X_interp + dX * mask_ft
    wav_hat = istft_complex_cfg(X_hat, hop=target_cplx_hop, length=wav_len, center=CPLX_CENTER)
    mel_hat = wav_to_bigvgan_mel_cfg(wav_hat, hop_size=target_mel_hop, center=CENTER_MEL)
    return X_hat, wav_hat, mel_hat

def roundtrip_hf_loss(mel_hat: torch.Tensor, mel_real: torch.Tensor, mask_mel: torch.Tensor):
    # Optional: compare after BigVGAN. Off by default in this notebook because it is expensive.
    sb = min(ROUNDTRIP_SUBBATCH, mel_hat.shape[0])
    mel_hat_sb = mel_hat[:sb]
    mel_real_sb = mel_real[:sb]
    mask_sb = mask_mel[:sb]

    wav_hat = mel_to_wave_bigvgan(mel_hat_sb)
    wav_real = mel_to_wave_bigvgan(mel_real_sb)

    mel_rt_hat = wav_to_bigvgan_mel(wav_hat)
    mel_rt_real = wav_to_bigvgan_mel(wav_real)

    T = min(mel_rt_hat.shape[-1], mel_rt_real.shape[-1], mask_sb.shape[-1])
    mel_rt_hat = mel_rt_hat[..., :T]
    mel_rt_real = mel_rt_real[..., :T]
    mask_sb = mask_sb[..., :T]

    return masked_l1(mel_rt_hat, mel_rt_real, mask_sb, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)


In [ ]:

# =========================
# Optimizer / checkpoint helpers
# =========================
opt_g = torch.optim.AdamW(G.parameters(), lr=LR_G, betas=(0.9, 0.99), weight_decay=WEIGHT_DECAY)

def save_ckpt(step: int):
    obj = {
        "step": step,
        "mode": MODE,
        "run_config": RUN_CONFIG,
        "G": G.state_dict(),
        "opt_g": opt_g.state_dict(),
        "hf_freq_weights": HF_FREQ_WEIGHTS.detach().cpu(),
        "mel_teacher_ckpt": str(MEL_TEACHER_CKPT),
        "complex_teacher_ckpt": str(COMPLEX_TEACHER_CKPT),
    }
    p = CKPT_DIR / f"{MODE}_step{step:07d}.pt"
    torch.save(obj, p)
    torch.save(obj, CKPT_DIR / "latest.pt")
    print("saved:", p)

start_step = 0
if RESUME_PATH is None and AUTO_RESUME_SAME_FAMILY:
    cands = sorted((DRIVE_ROOT / "checkpoints_runs").glob(f"{RUN_NAME}_*/latest.pt"))
    if cands:
        RESUME_PATH = cands[-1]

if RESUME_PATH is not None:
    RESUME_PATH = Path(RESUME_PATH)
    if RESUME_PATH.exists():
        ck = torch.load(str(RESUME_PATH), map_location="cpu")
        G.load_state_dict(ck["G"], strict=True)
        if "opt_g" in ck:
            opt_g.load_state_dict(ck["opt_g"])
        start_step = int(ck.get("step", 0))
        for pg in opt_g.param_groups:
            pg["lr"] = LR_G * RESUME_LR_SCALE
        print("Resumed fusion model from:", RESUME_PATH)
        print("start_step:", start_step)
    else:
        print("RESUME_PATH not found, starting fresh:", RESUME_PATH)
else:
    print("Starting mel-space gated fusion final fresh.")


In [ ]:

# =========================
# Smoke test: verifies shapes + differentiability before long training
# =========================
def one_batch_forward(wav: torch.Tensor, speed: float = None):
    wav = wav.clamp(-1.0, 1.0)
    pair = make_stretch_pair(wav, speed=speed)

    mel_real = pair["mel_real"]
    mel_interp = pair["mel_interp"]
    mask_mel = pair["mask_mel"]
    alpha_mel = pair["alpha_mel"]
    X_interp = pair["X_interp"]
    mask_cplx = pair["mask_cplx"]

    with torch.no_grad():
        mel_teacher_hat = mel_teacher_forward(mel_interp, mask_mel)
        _, wav_complex_hat, mel_complex_hat = complex_teacher_forward_stretch(
            X_interp,
            mask_cplx,
            wav_len=wav.shape[-1],
            target_cplx_hop=pair["target_cplx_hop"],
            target_mel_hop=pair["target_hop"],
        )
        T = min(mel_real.shape[-1], mel_interp.shape[-1], mel_teacher_hat.shape[-1], mel_complex_hat.shape[-1], mask_mel.shape[-1], alpha_mel.shape[-1])
        mel_real = mel_real[..., :T]
        mel_interp = mel_interp[..., :T]
        mel_teacher_hat = mel_teacher_hat[..., :T]
        mel_complex_hat = mel_complex_hat[..., :T]
        mask_mel = mask_mel[..., :T]
        alpha_mel = alpha_mel[..., :T]

    prior = make_hf_prior(mask_mel, n_mels=mel_real.shape[1], radius=PRIOR_RADIUS)
    gate_target = make_gate_target(mel_teacher_hat, mel_complex_hat, mel_real, prior)

    gate, delta = G(
        mel_interp=mel_interp,
        mel_teacher=mel_teacher_hat,
        mel_complex=mel_complex_hat,
        prior=prior,
        mask=mask_mel[:, :, 0, :],
        alpha=alpha_mel,
    )

    borrowed = gate * (mel_complex_hat - mel_teacher_hat)
    mel_fused = mel_teacher_hat + prior * (borrowed + delta)

    loss_recon = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=None, mask_dilate=MASK_DILATE)
    loss_hf = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
    loss_tdiff = masked_tdiff_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE)
    loss_anchor = anchor_l1(mel_fused, mel_real, mask_mel, freq_weights=None)
    loss_delta = ((delta.abs() * prior).sum() / (prior.sum() + 1e-8))
    loss_gate = ((gate.abs() * prior).sum() / (prior.sum() + 1e-8))
    loss_gate_align = (((gate - gate_target).abs() * prior).sum() / (prior.sum() + 1e-8))

    loss_rt = torch.zeros((), device=device)
    if USE_ROUNDTRIP_HF:
        loss_rt = roundtrip_hf_loss(mel_fused, mel_real, mask_mel)

    loss_total = (
        LAMBDA_RECON * loss_recon
        + LAMBDA_HF * loss_hf
        + LAMBDA_TDIFF * loss_tdiff
        + LAMBDA_ANCHOR_RECON * loss_anchor
        + LAMBDA_DELTA * loss_delta
        + LAMBDA_GATE * loss_gate
        + LAMBDA_GATE_ALIGN * loss_gate_align
        + LAMBDA_ROUNDTRIP_HF * loss_rt
    )

    return dict(
        loss_total=loss_total,
        loss_recon=loss_recon,
        loss_hf=loss_hf,
        loss_tdiff=loss_tdiff,
        loss_anchor=loss_anchor,
        loss_delta=loss_delta,
        loss_gate=loss_gate,
        loss_gate_align=loss_gate_align,
        loss_rt=loss_rt,
        mel_real=mel_real,
        mel_interp=mel_interp,
        mel_teacher_hat=mel_teacher_hat,
        mel_complex_hat=mel_complex_hat,
        mel_fused=mel_fused,
        gate=gate,
        prior=prior,
        mask_mel=mask_mel,
        alpha_mel=alpha_mel,
        speed=pair["speed"],
        target_hop=pair["target_hop"],
        wav_complex_hat=wav_complex_hat,
    )

if RUN_SMOKE_TEST_BEFORE_TRAINING:
    print("Running stretch-aware smoke test on one batch...")
    G.train()
    wav_smoke = next_mixed_batch()[:1]
    opt_g.zero_grad(set_to_none=True)
    out = one_batch_forward(wav_smoke, speed=0.5)
    out["loss_total"].backward()
    grad_norm = torch.sqrt(sum((p.grad.detach().float().norm() ** 2) for p in G.parameters() if p.grad is not None))
    opt_g.zero_grad(set_to_none=True)

    print("smoke loss:", float(out["loss_total"].detach().cpu()))
    print("grad norm:", float(grad_norm.detach().cpu()))
    print("speed:", out["speed"], "target_hop:", out["target_hop"])
    print("mel shapes real/interp/teacher/complex/fused:",
          tuple(out["mel_real"].shape), tuple(out["mel_interp"].shape),
          tuple(out["mel_teacher_hat"].shape), tuple(out["mel_complex_hat"].shape),
          tuple(out["mel_fused"].shape))
    print("inserted-frame fraction:", float(out["mask_mel"].mean().detach().cpu()))
    print("alpha mean inserted:", float((out["alpha_mel"].detach() * out["mask_mel"][:,0,0]).sum() / (out["mask_mel"][:,0,0].sum() + 1e-8)))
    print("gate mean in prior:", float((out["gate"].detach() * out["prior"]).sum() / (out["prior"].sum() + 1e-8)))
    assert torch.isfinite(out["loss_total"]).item(), "Smoke-test loss is not finite."
    assert grad_norm > 0, "No gradient reached the fusion model."
    print("Smoke test passed.")


In [ ]:

# =========================
# Train
# =========================
G.train()
t0 = time.time()

log_path = CKPT_DIR / "train_log.csv"
if not log_path.exists():
    log_path.write_text(
        "step,speed,target_hop,loss_total,loss_recon,loss_hf,loss_tdiff,loss_anchor,loss_delta,loss_gate,loss_gate_align,loss_roundtrip_hf,"
        "base_recon,ref_recon,base_hf,ref_hf,base_tdiff,ref_tdiff,avg_gate,borrow_hf,imp_hf_pct,insert_frac,minutes\n",
        encoding="utf-8",
    )

for step in range(start_step + 1, start_step + STEPS + 1):
    wav = next_mixed_batch()
    pair = make_stretch_pair(wav)

    mel_real = pair["mel_real"]
    mel_interp = pair["mel_interp"]
    mask_mel = pair["mask_mel"]
    alpha_mel = pair["alpha_mel"]
    X_interp = pair["X_interp"]
    mask_cplx = pair["mask_cplx"]

    with torch.no_grad():
        mel_teacher_hat = mel_teacher_forward(mel_interp, mask_mel)
        _, wav_complex_hat, mel_complex_hat = complex_teacher_forward_stretch(
            X_interp,
            mask_cplx,
            wav_len=wav.shape[-1],
            target_cplx_hop=pair["target_cplx_hop"],
            target_mel_hop=pair["target_hop"],
        )

        T = min(mel_real.shape[-1], mel_interp.shape[-1], mel_teacher_hat.shape[-1], mel_complex_hat.shape[-1], mask_mel.shape[-1], alpha_mel.shape[-1])
        mel_real = mel_real[..., :T]
        mel_interp = mel_interp[..., :T]
        mel_teacher_hat = mel_teacher_hat[..., :T]
        mel_complex_hat = mel_complex_hat[..., :T]
        mask_mel = mask_mel[..., :T]
        alpha_mel = alpha_mel[..., :T]

        base_recon = masked_l1(mel_teacher_hat, mel_real, mask_mel, freq_weights=None, mask_dilate=MASK_DILATE)
        base_hf    = masked_l1(mel_teacher_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
        base_tdiff = masked_tdiff_l1(mel_teacher_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE)

    prior = make_hf_prior(mask_mel, n_mels=mel_real.shape[1], radius=PRIOR_RADIUS)
    gate_target = make_gate_target(mel_teacher_hat, mel_complex_hat, mel_real, prior)

    gate, delta = G(
        mel_interp=mel_interp,
        mel_teacher=mel_teacher_hat,
        mel_complex=mel_complex_hat,
        prior=prior,
        mask=mask_mel[:, :, 0, :],
        alpha=alpha_mel,
    )

    borrowed = gate * (mel_complex_hat - mel_teacher_hat)
    mel_fused = mel_teacher_hat + prior * (borrowed + delta)

    loss_recon = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=None, mask_dilate=MASK_DILATE)
    loss_hf    = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
    loss_tdiff = masked_tdiff_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE)
    loss_anchor = anchor_l1(mel_fused, mel_real, mask_mel, freq_weights=None)

    loss_delta = ((delta.abs() * prior).sum() / (prior.sum() + 1e-8))
    loss_gate  = ((gate.abs()  * prior).sum() / (prior.sum() + 1e-8))
    loss_gate_align = (((gate - gate_target).abs() * prior).sum() / (prior.sum() + 1e-8))

    do_rt = USE_ROUNDTRIP_HF and (step % ROUNDTRIP_EVERY == 0)
    if do_rt:
        loss_roundtrip_hf = roundtrip_hf_loss(mel_fused, mel_real, mask_mel)
    else:
        loss_roundtrip_hf = torch.zeros((), device=device)

    loss_total = (
        LAMBDA_RECON * loss_recon
        + LAMBDA_HF * loss_hf
        + LAMBDA_TDIFF * loss_tdiff
        + LAMBDA_ANCHOR_RECON * loss_anchor
        + LAMBDA_DELTA * loss_delta
        + LAMBDA_GATE * loss_gate
        + LAMBDA_GATE_ALIGN * loss_gate_align
        + LAMBDA_ROUNDTRIP_HF * loss_roundtrip_hf
    )

    opt_g.zero_grad(set_to_none=True)
    loss_total.backward()
    torch.nn.utils.clip_grad_norm_(G.parameters(), 5.0)
    opt_g.step()

    with torch.no_grad():
        ref_recon = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=None, mask_dilate=MASK_DILATE)
        ref_hf    = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE)
        ref_tdiff = masked_tdiff_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=TDIFF_MASK_DILATE)
        imp_hf_pct = 100.0 * (base_hf.item() - ref_hf.item()) / max(1e-8, base_hf.item())
        avg_gate = float((gate * prior).sum().item() / (prior.sum().item() + 1e-8))
        borrow_hf = float(((borrowed.abs() * prior).sum() / (prior.sum() + 1e-8)).item())
        insert_frac = float(mask_mel.mean().item())

    if (step % LOG_EVERY == 0) or (step == start_step + 1):
        mins = (time.time() - t0) / 60.0
        row = [
            step,
            float(pair["speed"]),
            int(pair["target_hop"]),
            float(loss_total.item()),
            float(loss_recon.item()),
            float(loss_hf.item()),
            float(loss_tdiff.item()),
            float(loss_anchor.item()),
            float(loss_delta.item()),
            float(loss_gate.item()),
            float(loss_gate_align.item()),
            float(loss_roundtrip_hf.item()),
            float(base_recon.item()),
            float(ref_recon.item()),
            float(base_hf.item()),
            float(ref_hf.item()),
            float(base_tdiff.item()),
            float(ref_tdiff.item()),
            float(avg_gate),
            float(borrow_hf),
            float(imp_hf_pct),
            float(insert_frac),
            float(mins),
        ]
        with log_path.open("a", encoding="utf-8") as f:
            f.write(",".join(map(str, row)) + "\n")

        print(
            f"step {step:7d} | speed {pair['speed']:.2f} hop {pair['target_hop']} | "
            f"loss {loss_total.item():.4f} | "
            f"recon {loss_recon.item():.4f} | "
            f"hf {loss_hf.item():.4f} | "
            f"tdiff {loss_tdiff.item():.4f} | "
            f"anchor {loss_anchor.item():.4f} | "
            f"gate_align {loss_gate_align.item():.4f} | "
            f"gate {avg_gate:.3f} | "
            f"borrow_hf {borrow_hf:.4f} | "
            f"imp_hf {imp_hf_pct:.2f}%"
        )

    if (step % SAVE_EVERY == 0) or (step == start_step + STEPS):
        save_ckpt(step)

print("done. CKPT_DIR:", CKPT_DIR)


In [ ]:

# =========================
# Training curves
# =========================
log_df = pd.read_csv(CKPT_DIR / "train_log.csv")
display(log_df.tail())

plt.figure(figsize=(10,4))
for col in ["loss_total", "loss_hf", "loss_tdiff", "loss_anchor", "loss_gate_align"]:
    if col in log_df.columns:
        plt.plot(log_df["step"], log_df[col], label=col)
plt.legend(); plt.grid(True, alpha=0.3); plt.title(RUN_TAG + ": losses"); plt.show()

plt.figure(figsize=(10,4))
if "imp_hf_pct" in log_df.columns:
    plt.plot(log_df["step"], log_df["imp_hf_pct"], label="imp_hf_pct")
plt.axhline(0.0, ls="--", c="k", lw=1)
plt.legend(); plt.grid(True, alpha=0.3); plt.title(RUN_TAG + ": HF improvement over mel teacher"); plt.show()

plt.figure(figsize=(10,4))
for col in ["avg_gate", "borrow_hf"]:
    if col in log_df.columns:
        plt.plot(log_df["step"], log_df[col], label=col)
plt.legend(); plt.grid(True, alpha=0.3); plt.title(RUN_TAG + ": gate / borrowing"); plt.show()

plt.figure(figsize=(10,4))
if "speed" in log_df.columns:
    plt.scatter(log_df["step"], log_df["speed"], s=8, label="speed")
plt.legend(); plt.grid(True, alpha=0.3); plt.title(RUN_TAG + ": sampled stretch speeds"); plt.show()


In [ ]:

# =========================
# 6-second stretch-aware listening sanity check
# =========================
def read_split_lines_for_listen(split_name: str):
    for base in [SPLIT_DIR]:
        p = base / split_name
        if p.exists():
            lines = [ln.strip().strip('"').strip("'") for ln in p.read_text().splitlines()]
            return [Path(x) for x in lines if x]
    raise FileNotFoundError(f"Could not find split: {split_name}")

@torch.no_grad()
def eval_one_file_stretch(path: Path, speed: float = 0.5, seconds: float = EVAL_SEG_S, do_listen=True):
    G.eval()
    w = safe_load_wav_mono(path, SR)[: int(SR * seconds)].to(device)
    wb = w.unsqueeze(0)

    pair = make_stretch_pair(wb, speed=speed)
    mel_real = pair["mel_real"]
    mel_interp = pair["mel_interp"]
    mask_mel = pair["mask_mel"]
    alpha_mel = pair["alpha_mel"]

    mel_teacher_hat = mel_teacher_forward(mel_interp, mask_mel)
    _, wav_complex_hat, mel_complex_hat = complex_teacher_forward_stretch(
        pair["X_interp"],
        pair["mask_cplx"],
        wav_len=wb.shape[-1],
        target_cplx_hop=pair["target_cplx_hop"],
        target_mel_hop=pair["target_hop"],
    )

    T = min(mel_real.shape[-1], mel_interp.shape[-1], mel_teacher_hat.shape[-1], mel_complex_hat.shape[-1], mask_mel.shape[-1], alpha_mel.shape[-1])
    mel_real = mel_real[..., :T]
    mel_interp = mel_interp[..., :T]
    mel_teacher_hat = mel_teacher_hat[..., :T]
    mel_complex_hat = mel_complex_hat[..., :T]
    mask_mel = mask_mel[..., :T]
    alpha_mel = alpha_mel[..., :T]

    prior = make_hf_prior(mask_mel, n_mels=mel_real.shape[1], radius=PRIOR_RADIUS)
    gate, delta = G(mel_interp, mel_teacher_hat, mel_complex_hat, prior, mask_mel[:, :, 0, :], alpha_mel)
    mel_fused = mel_teacher_hat + prior * (gate * (mel_complex_hat - mel_teacher_hat) + delta)

    wav_interp = mel_to_wave_bigvgan(mel_interp)
    wav_teacher = mel_to_wave_bigvgan(mel_teacher_hat)
    wav_fused = mel_to_wave_bigvgan(mel_fused)
    wav_oracle = mel_to_wave_bigvgan(mel_real)

    base_hf = masked_l1(mel_teacher_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    ref_hf = masked_l1(mel_fused, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    interp_hf = masked_l1(mel_interp, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()
    complex_hf = masked_l1(mel_complex_hat, mel_real, mask_mel, freq_weights=HF_FREQ_WEIGHTS, mask_dilate=MASK_DILATE).item()

    print(f"{path}")
    print(f"speed={speed:.2f} | target_hop={pair['target_hop']} | frames={T} | inserted_frac={float(mask_mel.mean()):.3f}")
    print(f"HF L1 inserted: interp={interp_hf:.6f} | mel_teacher={base_hf:.6f} | complex_as_mel={complex_hf:.6f} | fused={ref_hf:.6f}")
    print("avg gate in prior:", float((gate.detach() * prior).sum() / (prior.sum() + 1e-8)))

    if do_listen:
        print("Original waveform (not stretched reference)")
        display(Audio(w.detach().cpu().numpy(), rate=SR))
        print("Oracle high-resolution mel through BigVGAN (upper bound-ish stretch target)")
        display(Audio(wav_oracle[0].detach().cpu().numpy(), rate=SR))
        print("Linear stretched mel through BigVGAN")
        display(Audio(wav_interp[0].detach().cpu().numpy(), rate=SR))
        print("Mel teacher through BigVGAN")
        display(Audio(wav_teacher[0].detach().cpu().numpy(), rate=SR))
        print("Complex teacher direct ISTFT (source duration, for artifact character only)")
        display(Audio(wav_complex_hat[0].detach().cpu().numpy(), rate=SR))
        print("Stretch-aware fused mel through BigVGAN")
        display(Audio(wav_fused[0].detach().cpu().numpy(), rate=SR))

    return dict(
        mel_real=mel_real,
        mel_interp=mel_interp,
        mel_teacher_hat=mel_teacher_hat,
        mel_complex_hat=mel_complex_hat,
        mel_fused=mel_fused,
        gate=gate,
        prior=prior,
        mask_mel=mask_mel,
        alpha_mel=alpha_mel,
    )

try:
    exact_paths = read_split_lines_for_listen(EXACT_LISTEN_SPLIT_NAME)
    exact_path = exact_paths[EXACT_LISTEN_SPLIT_INDEX]
    print("Exact listening case:", exact_path)
    for spd in EVAL_STRETCH_SPEEDS:
        print("\n" + "="*80)
        print(f"Stretch-aware evaluation, speed={spd}")
        _ = eval_one_file_stretch(exact_path, speed=float(spd), seconds=EVAL_SEG_S, do_listen=True)
except Exception as e:
    print("Exact listen failed, falling back to first speech validation file.")
    print("Reason:", repr(e))
    val_paths = read_list(speech_val)
    _ = eval_one_file_stretch(val_paths[0], speed=0.5, seconds=EVAL_SEG_S, do_listen=True)
